# Function 3 - Week 1: initial data exploration

Fifteen supplied observations, three inputs (`x1`, `x2` and `x3`), and larger outputs are better. Each input lies in `[0, 1)`: the domain is now a **cube**.

The [course description](https://classroom.emeritus.org/courses/18642/modules/items/3237303), recorded in the [Module 12 intake notes](../../../docs/module-12-intake.md), uses drug combinations and a transformed adverse-reaction score as its analogy. We maximise the supplied score; its negative sign is not an instruction to reverse the objective. The analogy does not establish the formula, a noise level or a known optimum.

**Taper-waper: start by reducing ignorance, finish by exploiting what we have learned.**

In [ ]:
from pathlib import Path
import sys

stage = next(p / "stage-2-bbo" for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "stage-2-bbo/src").is_dir())
sys.path.insert(0, str(stage / "src"))
from initial_exploration import load_initial_data, preview, plot_ordered_outputs
from multidimensional_exploration import (plot_coordinate_pairs, plot_cube_slices,
    plot_outputs_by_input, correlation_sensitivity, exploration_metrics)

data = load_initial_data(stage, function_id=3)
figures = stage.parent / ".runtime/function-3-initial-analysis"
metrics = exploration_metrics(data, save_to=figures / "exploration-summary.json")

## A quick look at the values

All fifteen observations, in their supplied order, rounded to six decimal places for display. **Rank 1 is the highest observed output; rank 15 is the lowest.** These are output ranks, not row IDs.

In [ ]:
preview(data, rows=len(data), output_format="{:.6f}")

Every output is negative. They range from **−0.398926 to −0.034835**, so the best supplied result is the one closest to zero. That does not establish zero as the function's ceiling.

## Where the observations sit

Three inputs produce **three coordinate pairs**: `(x1, x2)`, `(x1, x3)` and `(x2, x3)`. Each panel projects the same fifteen observations onto one pair. Marker numbers and colours retain the same output ranks throughout.

Two markers can be close in one projection but far apart in the omitted coordinate. These panels show the ordering of observed values; they do not interpolate an output surface between them.

In [ ]:
plot_coordinate_pairs(data, save_to=figures / "ranked-locations.png")

The highest output is **−0.034835 at (0.492581, 0.611593, 0.340176)**. The runner-up, **−0.036378 at (0.600097, 0.725136, 0.066089)**, is about **0.316** units away in the cube. Their outputs are close, but their third coordinates differ substantially.

Ranks 3 and 4 sit elsewhere, at lower `x1` and `x2`. The best observations therefore do not identify a single compact pocket.

The closest pair in the **full three-dimensional space** is ranks **3 and 11**, separated by **0.116** input units. Their outputs differ by **0.0652**, despite looking fairly close geometrically. With all three inputs changing, that cannot isolate an individual input's effect or distinguish local variation from noise.

## Sampling gaps in the projections

Brighter areas are farther from the nearest projected observation. The three panels share a distance colour scale. These are geometric distances in each **two-input plane**, not predicted outputs or model uncertainty.

In [ ]:
plot_coordinate_pairs(data, gaps=True, save_to=figures / "projected-sampling-gaps.png")

No supplied observation has **`x2 < 0.219917`**. That leaves a slab covering roughly **22% of the cube's volume** with no observation inside it. It does not mean that the rest of the cube is well covered.

There is also **no observation with both `x2 > 0.6` and `x3 > 0.6`**, whatever `x1` is. This is a missing combination of inputs, not just a missing value of one input.

Projection always makes distances no larger than their full-space counterparts. A dark patch can therefore look reassuring while concealing a substantial gap in the omitted direction.

## Looking through the cube

These panels are **slices**, not projections: `x3` is fixed at 0.1, 0.5 and 0.9. At every displayed `(x1, x2)`, the colour is the distance from the resulting **three-coordinate point** to its nearest actual observation, allowing that observation to lie anywhere in the cube.

All three slices share one colour scale. Observations are not overlaid because none lies exactly on these planes; projecting them here would disguise their distance in `x3`. The scale differs from the preceding projected-distance maps.

In [ ]:
plot_cube_slices(data, save_to=figures / "cube-sampling-gaps.png")

The high-`x3` slice exposes a broad gap towards high `x2`. At lower `x3`, the pattern of gaps changes. A two-coordinate query cannot specify which of these situations we are testing: every new query needs all three coordinates.

A uniform **81 × 81 × 81** grid through the cube gives a mean nearest-observation distance of about **0.255**; roughly 10% of its locations are more than **0.396** units from a sample. The mean from a coarser 41 × 41 × 41 calculation agrees at three decimal places. These summarise spatial coverage, not the probability of missing a good response. Three selected slices alone would not measure coverage of the entire cube.

## How large are the outputs?

The original scale keeps the objective clear: higher, less-negative values are better. The second panel shows `log₁₀ |y|` for comparison with the earlier functions. **Here every output is negative, so a higher logarithmic magnitude means a worse observed result.** The logarithm is a display aid, not a replacement objective.

In [ ]:
plot_ordered_outputs(data, save_to=figures / "ordered-outputs.png")

The largest magnitude is about **11.45 times** the smallest, just over one order of magnitude. Rank 15, **−0.398926**, is separated from the other fourteen observations, which lie between **−0.131461 and −0.034835**.

The following pair of charts omits rank 15 **from this view only**. The original-scale panel zooms in; the logarithmic panel retains the preceding limits. The observation remains in the dataset and all coverage calculations.

In [ ]:
plot_ordered_outputs(data, exclude_rank=15, save_to=figures / "ordered-outputs-without-extreme.png")

The remaining values are easy to distinguish on their original scale. There is no need for a transformation just to make them readable, and no evidence that rank 15 is an error to discard.

## Does one input explain the outputs?

These scatter plots use the original output scale and shared vertical limits. Only ranks 1, 2 and 15 are numbered to keep overlapping points legible; all observations are present.

In [ ]:
plot_outputs_by_input(data, save_to=figures / "outputs-by-input.png")

The three observations with `x3 > 0.75` are ranks **12, 13 and 15**. That makes high `x3` a plausible concern, but most lower-`x3` observations are not especially good either. Different `x1` and `x2` settings could change the effect.

The worst result has **`x3 = 0.990882`**. Another observation at **`x3 = 0.919771`** returns **−0.113869**: still low-ranked, but much less negative. The existing observations do not establish a simple rule that increasing `x3` always makes the score worse.

A short sensitivity check shows how much rank 15 influences that impression. **Pearson correlation** measures straight-line association; **Spearman correlation** measures whether the ordering tends to rise or fall together. These are descriptive summaries, not isolated effects or significance tests.

In [ ]:
correlation_sensitivity(data)

For `x3`, Pearson correlation moves from about **−0.57 to −0.14** when rank 15 is omitted from the calculation. The rank-based association is also weaker, about **−0.23** with all fifteen observations. The apparent strong downward trend is therefore sensitive to one result. This warrants testing the explanation, not deleting the observation.

## What this means for the taper

The first risk to reduce is **committing too early to a simple explanation of `x3`**. The data leave open a broad high-`x3` penalty, an interaction with the other coordinates, or a more local dip around the worst observation. There are no repeat evaluations to measure noise directly.

Coverage also has a more demanding meaning now. A point can fill a gap in one projection while adding little in another; any coverage score used to choose it must use **all three coordinates together**. The empty high-`x2`, high-`x3` region is one concrete blind spot to weigh against testing near the extreme observation or refining near the best results.

Knowledge gradient could compare the expected value of those answers once plausible response models have been checked. At this stage, the supplied data support the questions more strongly than any particular next coordinate. No model has been fitted or next query selected in this exploration.

There are only three pairwise panels here; eight inputs would produce **28**. More panels will not remove the missing-coordinate problem. Full-space coverage calculations and explicit hypotheses about interactions become increasingly useful, while differences between functions and starting sample sizes prevent attributing every change to dimensionality alone.

The best supplied value, **−0.034835**, is a reference point for improvement, not evidence that we are near the optimum. Early queries should challenge consequential assumptions; later observations can justify concentrating on a promising region.